In [1]:
import os
import gc
import zipfile
import pandas as pd
import librosa
import numpy as np
from PIL import Image
from tqdm.auto import tqdm

# --- 1. Cấu hình Đường dẫn ---
print("Đang thiết lập đường dẫn...")
# (Đã cập nhật cho máy Windows cục bộ)

# Đường dẫn đầu vào (Input)
AUDIO_BASE_PATH = r'D:\Python_mohinh\data\FMA_Small\fma_small\fma_small'
METADATA_PATH = r'D:\Python_mohinh\data\MASTER_METADATA_FMA_ONLY_Chunks3.csv'

# Đường dẫn đầu ra (Output) - ĐÃ THÊM 'r'
OUTPUT_IMAGE_DIR = r"D:\Python_mohinh\data\mel_spectrograms_db_30s"
OUTPUT_CSV_PATH = r"D:\Python_mohinh\data\mel_metadata_new_30s.csv"
OUTPUT_ZIP_PATH = r"D:\Python_mohinh\data\fma_mel_spectrograms_db_30s.zip" # File ZIP cuối cùng

# Tạo thư mục chứa ảnh
os.makedirs(OUTPUT_IMAGE_DIR, exist_ok=True)

# --- 2. Định nghĩa Hàm xử lý Âm thanh (Chất lượng cao - 30s) ---

def create_mel_30s_spectrogram(audio_path, 
                               track_id_str, 
                               out_dir, 
                               sr=22050, 
                               n_mels=128, 
                               n_fft=2048, 
                               hop_length=512,
                               duration=30.0):
    """
    Tải 1 file audio (30s), tạo 1 Mel Spectrogram (dB) duy nhất
    và lưu dưới dạng ảnh JPEG.
    
    Trả về: Tên file ảnh đã được tạo.
    """
    
    try:
        # 1. Tải file âm thanh (tối đa 30s)
        y, sr_loaded = librosa.load(audio_path, sr=sr, mono=True, duration=duration)
        
        # 2. Đệm (Pad) hoặc Cắt (Cut) để đảm bảo đúng 30s
        target_samples = int(duration * sr) # 30s * 22050 = 661500
        if len(y) < target_samples:
            y = np.pad(y, (0, target_samples - len(y)), 'constant')
        else:
            y = y[:target_samples]

        # 3. Tạo Mel Spectrogram
        S = librosa.feature.melspectrogram(y=y, sr=sr, n_mels=n_mels, n_fft=n_fft, hop_length=hop_length)
        
        # 4. Chuyển sang Decibel (dB) - CỰC KỲ QUAN TRỌNG
        S_db = librosa.power_to_db(S, ref=np.max)
        
        # 5. Chuẩn hóa (Normalize) về dải 0-1 để lưu ảnh
        S_db_clipped = np.clip(S_db, -80, 0)
        S_db_normalized = (S_db_clipped + 80) / 80 # Giờ ảnh nằm trong dải [0, 1]
        
        # 6. Chuyển sang 8-bit image (0-255)
        # Kích thước ảnh sẽ là (128, 1292)
        img_data = (S_db_normalized * 255).astype(np.uint8)
        
        # 7. Tạo ảnh 3 kênh (RGB) từ mảng numpy và lưu
        img = Image.fromarray(img_data).convert('RGB')
        
        # 8. Đặt tên file ảnh mới
        new_filename = f"{track_id_str}_mel_db_30s.jpg"
        out_image_path = os.path.join(out_dir, new_filename)
        img.save(out_image_path, 'JPEG', quality=95) # Lưu chất lượng cao
            
        return new_filename
    
    except Exception as e:
        print(f"Lỗi khi xử lý {track_id_str} (file: {audio_path}): {e}")
        return None

# --- 3. Vòng lặp Chính: Xử lý Dữ liệu ---
print("Đang tải file metadata gốc...")
try:
    # (Quan trọng: ép track_id_str là string để giữ các số 0 ở đầu)
    df = pd.read_csv(METADATA_PATH, dtype={'track_id_str': str})
except Exception as e:
    print(f"LỖI: Không thể đọc file metadata tại: {METADATA_PATH}")
    print("Bạn đã upload file `MASTER_METADATA_FMA_ONLY_Chunks3.csv` lên /content/ chưa?")
    raise

# Lấy 8000 track_id duy nhất và thể loại của chúng
df_unique = df[['track_id_str', 'genre']].drop_duplicates().reset_index(drop=True)

print(f"Đã tìm thấy {len(df_unique)} track âm thanh duy nhất để xử lý.")

new_metadata_rows = [] # Nơi lưu metadata mới

print("Bắt đầu quá trình tạo Mel Spectrogram (dB) 30 giây...")
print("CẢNH BÁO: Quá trình này sẽ RẤT LÂU vì đọc từ Google Drive.")

# Dùng tqdm để xem thanh tiến trình
for row in tqdm(df_unique.itertuples(), total=len(df_unique)):
    track_id_str = row.track_id_str # Đã là string
    genre = row.genre
    
    # Xây dựng đường dẫn file MP3
    # track_id_str (ví dụ: '000002') -> folder ('000')
    folder = track_id_str.zfill(6)[:3]
    filename = f"{track_id_str.zfill(6)}.mp3"
    audio_path = os.path.join(AUDIO_BASE_PATH, folder, filename)
    
    if not os.path.exists(audio_path):
        # In ra 10 lỗi đầu tiên, sau đó im lặng
        if 'missing_file_count' not in locals(): missing_file_count = 0
        missing_file_count += 1
        if missing_file_count < 10:
             print(f"Cảnh báo: Không tìm thấy file {audio_path}. Bỏ qua.")
        continue
        
    # Tạo 1 ảnh 30s và lưu nó
    new_filename = create_mel_30s_spectrogram(
        audio_path=audio_path,
        track_id_str=track_id_str, # Truyền ID (ví dụ: '000002')
        out_dir=OUTPUT_IMAGE_DIR
    )
    
    # Thêm thông tin vào metadata mới
    if new_filename:
        new_metadata_rows.append({
            'filename': new_filename,
            'track_id_str': track_id_str,
            'genre': genre
        })
    
    # Giải phóng bộ nhớ sau mỗi 100 file
    if row.Index % 100 == 0:
        gc.collect()

print("Đã hoàn thành tạo ảnh.")
if 'missing_file_count' in locals() and missing_file_count > 10:
    print(f"Tổng cộng có {missing_file_count} file MP3 không tìm thấy (đã bỏ qua).")

# --- 4. Tạo file CSV mới ---
print("Đang tạo file metadata CSV mới...")
new_df = pd.DataFrame(new_metadata_rows)
if new_df.empty:
    print("LỖI NGHIÊM TRỌNG: Không có ảnh nào được tạo.")
    print(f"Hãy kiểm tra lại đường dẫn AUDIO_BASE_PATH của bạn: {AUDIO_BASE_PATH}")
else:
    new_df.to_csv(OUTPUT_CSV_PATH, index=False)
    print(f"Đã lưu metadata mới tại: {OUTPUT_CSV_PATH}")

    # --- 5. Nén (Zip) Dữ liệu ---
    print("Bắt đầu nén file ZIP...")
    try:
        with zipfile.ZipFile(OUTPUT_ZIP_PATH, 'w', zipfile.ZIP_DEFLATED) as zipf:
            
            # 1. Thêm file CSV
            print(f"- Thêm {OUTPUT_CSV_PATH} vào file zip...")
            zipf.write(OUTPUT_CSV_PATH, arcname=os.path.basename(OUTPUT_CSV_PATH))
            
            # 2. Thêm tất cả các file ảnh
            print(f"- Thêm ảnh từ {OUTPUT_IMAGE_DIR} vào file zip...")
            
            # Đặt tên thư mục con trong file zip
            image_zip_folder = "mel_spectrograms_db_30s"
            
            for filename in tqdm(new_df['filename'], total=len(new_df)):
                image_path = os.path.join(OUTPUT_IMAGE_DIR, filename)
                if os.path.exists(image_path):
                    # arcname là đường dẫn bên trong file zip
                    zip_arc_path = os.path.join(image_zip_folder, filename)
                    zipf.write(image_path, arcname=zip_arc_path)

        print("\n--- HOÀN TẤT! ---")
        print(f"File ZIP của bạn đã sẵn sàng tại:")
        print(f"{OUTPUT_ZIP_PATH}")
        print(f"Tổng dung lượng file: {os.path.getsize(OUTPUT_ZIP_PATH) / (1024*1024):.2f} MB")
        print("Bạn có thể tải về từ thanh 'Files' bên trái của Colab.")

    except Exception as e:
        print(f"LỖI trong quá trình nén file: {e}")

Đang thiết lập đường dẫn...
Đang tải file metadata gốc...
Đã tìm thấy 7994 track âm thanh duy nhất để xử lý.
Bắt đầu quá trình tạo Mel Spectrogram (dB) 30 giây...
CẢNH BÁO: Quá trình này sẽ RẤT LÂU vì đọc từ Google Drive.


  0%|          | 0/7994 [00:00<?, ?it/s]

C:\Users\Trinh Gia Bao\AppData\Local\Temp\ipykernel_26848\1746087640.py:45: UserWarning: PySoundFile failed. Trying audioread instead.
  y, sr_loaded = librosa.load(audio_path, sr=sr, mono=True, duration=duration)
d:\Python_mohinh\.venvrun\Lib\site-packages\librosa\core\audio.py:184: FutureWarning: librosa.core.audio.__audioread_load
	Deprecated as of librosa version 0.10.0.
	It will be removed in librosa version 1.0.
  y, sr_native = __audioread_load(path, offset, duration, dtype)


Đã hoàn thành tạo ảnh.
Đang tạo file metadata CSV mới...
Đã lưu metadata mới tại: D:\Python_mohinh\data\mel_metadata_new_30s.csv
Bắt đầu nén file ZIP...
- Thêm D:\Python_mohinh\data\mel_metadata_new_30s.csv vào file zip...
- Thêm ảnh từ D:\Python_mohinh\data\mel_spectrograms_db_30s vào file zip...


  0%|          | 0/7994 [00:00<?, ?it/s]


--- HOÀN TẤT! ---
File ZIP của bạn đã sẵn sàng tại:
D:\Python_mohinh\data\fma_mel_spectrograms_db_30s.zip
Tổng dung lượng file: 615.46 MB
Bạn có thể tải về từ thanh 'Files' bên trái của Colab.
